# 11 — The "blank sheet" environment (step 9a)

So far the agent drew on a 5×5 grid, with ready-made segments and 17 "buttons". Here we remove these aids:
the agent draws **freely on a 28×28 pixel canvas**. As in notebook 01, we only build and check the environment;
no learning yet.

## What changes

| | 5×5 grid (notebooks 01–10) | Blank sheet (`src/free_draw.py`) |
|---|---|---|
| Canvas | 25 points | 784 pixels (28 × 28) |
| Target | a list of segments | an **image** of the letter |
| Moves | 1 cell in 8 directions | **16 directions × 3 lengths (1, 2, 5 pixels)** → 40 distinct moves |
| Actions | 8 × 2 + STOP = 17 | 40 × 2 (pen up / pen down) + STOP = **81** |
| Reward | +1 per correct segment | **resemblance** between the drawing and the image |
| What the agent sees | 170 or 266 numbers | **4 images** of 28 × 28 (target, drawing, pen position, pen down) |

## How a drawing is judged (with a tolerance of 1 pixel)

The letter is drawn as 1-pixel lines: its **skeleton**. A human never draws exactly on the line, so:
- a skeleton pixel is **covered** when the agent has inked a pixel at most 1 pixel away;
- an inked pixel is **good** when it lies in the **zone** (the skeleton widened by 1 pixel).

## Rewards (same spirit as the grid)

| Event | Reward | Note |
|---|---|---|
| Newly covered skeleton pixel | +0.2 | a grid segment is 5 pixels long → about +1 per segment, as before |
| Newly inked pixel outside the zone | −0.2 | scribbling does not pay |
| Each move | −0.05 | |
| Starting a new stroke | −0.3 | |
| Trying to leave the canvas | −0.2 | the pen does not move |
| STOP | +2 if ≥ 95 % of the skeleton is covered, and −0.2 per skeleton pixel still uncovered | |

Episodes start on a natural starting point of the letter (pen up) and are cut after 60 steps.

In [ ]:
import sys
sys.path.insert(0, "..")

import numpy as np
import matplotlib.pyplot as plt

from src.letters import make_letter, TRAINING_LETTERS, TEST_LETTERS
from src.free_draw import (FreeDrawEnv, letter_image, scripted_actions, action_name,
                           MOVES, MOVE_LABELS, N_ACTIONS, N_MOVES, STOP, SIZE)
from src.visualization import draw_free_episode, SURFACE, TEXT, MUTED

ALL_LETTERS = TRAINING_LETTERS + TEST_LETTERS
print(f"canvas {SIZE} x {SIZE},  {N_MOVES} distinct moves,  {N_ACTIONS} actions")

## 1. The 26 letters as target images

In [ ]:
fig, axes = plt.subplots(3, 9, figsize=(13, 4.8), facecolor=SURFACE)
for ax, name in zip(axes.ravel(), ALL_LETTERS):
    env = FreeDrawEnv(name)
    draw_free_episode(ax, env.target, env.zone, [], title=f"{name} ({env.n_target} px)")
axes.ravel()[-1].axis("off")
fig.text(0.5, 0.01, "grey: skeleton (target)   ·   very light: tolerance zone (skeleton widened by 1 pixel)   ·   (number of skeleton pixels)",
         ha="center", fontsize=9, color=MUTED)
fig.tight_layout(rect=(0, 0.04, 1, 1))
fig.savefig("../results/11_targets.png", dpi=150, facecolor=SURFACE)
plt.show()

## 2. The moves

All 40 distinct moves, drawn from the centre of the canvas. Short moves (1 and 2 pixels) are for fine
adjustments; long moves (5 pixels) cover the distance between two points of the old grid in one action
(some short moves round to the same pixel, which is why there are 40 and not 48).

In [ ]:
fig, ax = plt.subplots(figsize=(4, 4), facecolor=SURFACE)
centre = (SIZE // 2, SIZE // 2)
colors = {1: "#eb6834", 2: "#1baf7a", 5: "#2a78d6"}
for (dr, dc), (angle, L) in zip(MOVES, MOVE_LABELS):
    ax.annotate("", xy=(centre[1] + dc, centre[0] + dr), xytext=(centre[1], centre[0]),
                arrowprops=dict(arrowstyle="->", color=colors[L], lw=1.2))
ax.set_xlim(centre[1] - 7, centre[1] + 7)
ax.set_ylim(centre[0] + 7, centre[0] - 7)
ax.set_aspect("equal")
ax.set_xticks([]); ax.set_yticks([])
ax.legend([plt.Line2D([], [], color=c, lw=1.5) for c in colors.values()], [f"{L} px" for L in colors],
          frameon=False, fontsize=9, loc="upper left")
ax.set_title("The 40 distinct moves (pixels)", fontsize=10, color=TEXT)
plt.show()

## 3. A random agent

In [ ]:
env = FreeDrawEnv("E", seed=0)
rng = np.random.default_rng(0)
env.reset()
total, done = 0.0, False
while not done:
    _, r, terminated, truncated, _ = env.step(int(rng.integers(N_ACTIONS)))
    total += r
    done = terminated or truncated
print(f"steps {env.steps}, total reward {total:.2f}, coverage {env.coverage():.0%}, precision {env.precision():.0%}, "
      f"ended by {'STOP' if terminated else 'step limit'}")
fig, ax = plt.subplots(figsize=(3.2, 3.2), facecolor=SURFACE)
draw_free_episode(ax, env.target, env.zone, env.history, "Random agent on E")
plt.show()

## 4. Checking the rules

Hand-computed values, as in notebook 01 (`L` has 36 skeleton pixels).

In [ ]:
def play(letter, start, actions):
    env = FreeDrawEnv(letter)
    env.reset(start=start)
    total, terminated, truncated = 0.0, False, False
    for a in actions:
        _, r, terminated, truncated, _ = env.step(a)
        total += r
    return total, env, terminated, truncated

down5, right5 = MOVES.index((5, 0)), MOVES.index((0, 5))
up1, left1, right1, down1 = MOVES.index((-1, 0)), MOVES.index((0, -1)), MOVES.index((0, 1)), MOVES.index((1, 0))
start_L = (4, 4)   # top of the L

# (a) the whole L with 5-pixel moves, then STOP: 36 pixels x 0.2 - 0.3 (stroke) - 7 x 0.05 + 2 (bonus)
total, env, *_ = play("L", start_L, [N_MOVES + down5] * 4 + [N_MOVES + right5] * 3 + [STOP])
assert np.isclose(total, 36 * 0.2 - 0.3 - 7 * 0.05 + 2) and env.coverage() == 1 and env.precision() == 1
print(f"(a) whole L then STOP:            {total:+.2f}   coverage {env.coverage():.0%}, precision {env.precision():.0%}")

# (b) a stroke far from the letter: 6 new pixels outside the zone (start pixel + 5)
total, env, *_ = play("L", start_L, [right5, right5, N_MOVES + down5])
assert np.isclose(total, 2 * -0.05 + (-0.05 - 0.3 + 6 * -0.2)), total
print(f"(b) stroke outside the letter:    {total:+.2f}")

# (c) tolerance: drawing 1 pixel to the right of the vertical bar still covers it
total, env, *_ = play("L", start_L, [right1] + [N_MOVES + down5] * 4)
print(f"(c) drawn 1 pixel beside the bar: coverage {env.coverage():.0%} of the L (the vertical bar is fully covered), "
      f"precision {env.precision():.0%}")
assert env.precision() == 1 and env.covered[4:25, 4].all()

# (d) leaving the canvas: the pen stays, -0.2
total, env, *_ = play("L", (0, 0), [up1])
assert np.isclose(total, -0.2) and env.pen == (0, 0)
print(f"(d) move off the canvas:          {total:+.2f}")

# (e) STOP immediately: -0.2 x 36 uncovered pixels
total, env, *_ = play("L", start_L, [STOP])
assert np.isclose(total, -0.2 * 36)
print(f"(e) STOP immediately:             {total:+.2f}")

# (f) step limit: truncated after 60 steps, not terminated
total, env, terminated, truncated = play("L", (0, 0), [up1] * 60)
assert truncated and not terminated
print("(f) step limit: truncated after 60 steps, not terminated")
print("\nAll checks passed.")

## 5. Can every letter be drawn perfectly with these moves?

Before asking an agent to learn, we check that the task is possible: a hand-written policy follows the strokes
of each letter with 5-pixel moves (pen up between strokes), then presses STOP.

In [ ]:
panels, rows = [], []
for name in ALL_LETTERS:
    env = FreeDrawEnv(name)
    env.reset(start=env.start_pixels[0])
    total, terminated = 0.0, False
    for a in scripted_actions(env.letter):
        _, r, terminated, truncated, _ = env.step(a)
        total += r
    rows.append((name, env.steps, env.coverage(), env.precision(), total, env.perfect(terminated)))
    panels.append((env, f"{name} ({env.steps} actions)"))
print(f"{'letter':<7}{'actions':>8}{'coverage':>10}{'precision':>11}{'reward':>8}{'perfect':>9}")
for name, steps, cov, prec, total, ok in rows:
    print(f"{name:<7}{steps:>8}{cov:>10.0%}{prec:>11.0%}{total:>8.2f}{str(ok):>9}")
print(f"\nperfect drawings: {sum(r[-1] for r in rows)} / {len(rows)}")

fig, axes = plt.subplots(3, 9, figsize=(13, 4.8), facecolor=SURFACE)
for ax, (env, title) in zip(axes.ravel(), panels):
    draw_free_episode(ax, env.target, env.zone, env.history, title)
axes.ravel()[-1].axis("off")
fig.text(0.5, 0.01, "blue: inked in the zone   ·   red: inked outside   ·   dashed: pen up   ·   orange ring: start",
         ha="center", fontsize=9, color=MUTED)
fig.tight_layout(rect=(0, 0.04, 1, 1))
fig.savefig("../results/11_scripted_drawings.png", dpi=150, facecolor=SURFACE)
plt.show()

## 6. What the agent will see

The observation is made of 4 images of 28 × 28. Here, in the middle of an episode on E (after 5 moves of the
hand-written policy). The learning agent (step 9b) will read them with a convolutional network (CNN), which
applies the same small filters everywhere in the image: the natural tool for a *local* rule such as "is there
a missing piece of the letter next to my pen?" (lesson of step 7c).

In [ ]:
env = FreeDrawEnv("E")
env.reset(start=env.start_pixels[0])
for a in scripted_actions(env.letter)[:5]:
    obs, *_ = env.step(a)
fig, axes = plt.subplots(1, 4, figsize=(10, 2.8), facecolor=SURFACE)
for ax, img, title in zip(axes, obs, ["1. target", "2. drawing so far", "3. pen position", "4. pen down (1 everywhere)"]):
    ax.imshow(img, cmap="Greys", vmin=0, vmax=1, interpolation="nearest")
    ax.set_title(title, fontsize=10, color=TEXT)
    ax.set_xticks([]); ax.set_yticks([])
fig.tight_layout()
plt.show()
print("observation shape:", obs.shape)

## Summary

- The blank sheet has no ready-made segments and no grid: 40 free moves, pen up or down, and an image as target.
- A drawing is judged with a tolerance of 1 pixel (coverage of the skeleton, precision of the ink).
- The reward keeps the spirit of the grid (about +1 per 5-pixel piece of letter, −0.3 per stroke, +2 at STOP).
- Every rule gives the hand-computed value, and **all 26 letters can be drawn perfectly** with these moves.
- Next step (9b): a DQN with a convolutional network learns to draw on this sheet.